In [ ]:
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import warnings
warnings.filterwarnings('ignore')



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
df_path = os.path.join(path, 'Q3_data.csv')
df = pd.read_csv(df_path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 1: Write your code here:
#We use this code to figure out the count of missing values
print("Missing values:")
print(df.isnull().sum())

df.drop(columns=['D_142'])

numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("Target")

for col in df[numerical_cols]:
  df[col] = df[col].fillna(df[col].dropna().mean())
df.head()



In [ ]:
# Task 2: Write your code here:
print("Checking for duplicate rows...")
duplicate_rows = df.duplicated().sum()
if duplicate_rows > 0:
    print(f"Found {duplicate_rows} duplicate rows. Removing them...")
    df.drop_duplicates(inplace=True)
    print("Duplicate rows removed.")
else:
    print("No duplicate rows found.")

In [ ]:
# Task 3: Write your code here:
categorical_cols = df.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))
#no categorical columns

In [ ]:
# Task 4: Write your code here:
numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("Target")

scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
df.head()

In [ ]:
# Task 5: Write your code here:
import seaborn as sns
def check_target_imbalance(df, target_column):
  print("Target Distribution:")
  print(df[target_column].value_counts(normalize=True))
  sns.countplot(x=df[target_column])
  plt.title("Target Distribution")
  plt.show()

check_target_imbalance(df, "Target")

#Target is imbalanced as can be seen in the graph

In [ ]:
# Task 1: Write your code here:
X = df.drop("Target", axis=1).astype(float)
y = df['Target']


In [ ]:
from catboost.core import CatBoostClassifier
# Task 2,3,4,5: Write your code here:
%pip install kagglehub catboost lightgbm tqdm -q
from catboost import CatBoostRegressor
from sklearn.metrics import f1_score
model = CatBoostClassifier(verbose=0)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

f1_scores = []
for fold, (train_index, test_index) in enumerate(skf.split(X, y), start=1):

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # Train
  model.fit(X_train, y_train)

  # Predict
  y_pred = model.predict(X_test)

  # Calculate metrics
  #f1 = f1_score(y_test, y_pred)

  # Store results
  #f1_scores.append(f1)

#print(sum(f1_scores)/len(f1_scores))

In [ ]:
# Task 1: Write your code here:
sklearn_models = {
  "CatBoost": CatBoostClassifier(
      verbose=0,
      n_estimators=320,
      max_depth=4
  )
}
importances = {}
importances['CatBoost'] = sklearn_models['CatBoost'].feature_importances_

# Create a 1x3 plot
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes = axes.flatten()
features = X.columns

for i, (model_name, imp) in enumerate(importances.items()):
  # Sort features by importance for a cleaner plot
  sorted_idx = np.argsort(imp)

  ax = axes[i]
  ax.barh(features[sorted_idx], imp[sorted_idx])
  ax.set_title(f"{model_name} Feature Importance")
  ax.set_xlabel("Importance Score")

plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:
#The most important feature according to the graph above is

In [ ]:
# Task Bonus: Write your code here: